
# Transform Constructors Data

1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constrctorId -> constructor_id)
4. Rename columns to make them more meaningful (name -> constructor_name)
5. Remove duplicate records
6. Transform values of columns nationality to Title Case
7. Write the transformed data to silver constructors table

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"


### Step 1 - Read bronze constructors table

In [0]:
# circuits_df = spark.table(bronze_table)
constructors_df = spark.read.table(bronze_table).filter((col("batch_id") == v_batch_id))

### Step 2 - Keep only the columns required for analytics (Drop url column)



In [0]:
constructors_selected_df = constructors_df.drop(
    col("url")   
)


### Step 3

- Standardise column names using snake_case (constructorsId -> constructors_id)
- Rename columns to make them more meaningful (name -> constructors_name)

In [0]:
constructors_renamed_df = constructors_selected_df\
    .withColumnsRenamed(
        {
            "constructorId":"constructor_id",
            "name":"constructor_name"
        }
    )


### Step 5 - Remove Duplicate Records


In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])


### Step  6 - Transform Value of column nationality to Title Case


In [0]:
constructors_final_df = (
    constructors_distinct_df
    .withColumn('nationality', initcap('nationality'))
)



### Step  7 - Write the transformed data to silver constructors table

In [0]:
write_to_silver(
    constructors_final_df,
    silver_table,
    "t.constructor_id = s.constructor_id" ,
    [
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))